![image](Banner.jpg)

# Bandidos con recompensas no estacionarias

Para este ejercicio extenderemos la implementación de los bandidos con la estrategia de epsilon decay para permitir que el bandido maneje recompensas variables.

El cálculo del valor esperado de las recompensas para cada uno de los brazos se hace con la fórmula 
$$
Q_{n+1} = (1-\alpha)^nQ_1 + \sum\limits_{i=1}^n \alpha(1-\alpha)^{n-i}r_i
$$

donde $Q_1$ es la estimación inicial para el valor del brazo, $\alpha$ es el valor del paso y $r_i$ es la recompensa obtenida en cada una de las ocurrencias del brazo.


El código base de este ejercicio será la implementación del bandido de k-brazos con &epsilon;-decay.

Los cambios que realizaremos sobre el bandido se definen incrementalmente, pero todos se deben realizar sobre la misma celda del notebook. Cada modificación requerida estará asociada con sus pruebas correspondientes.


### Definición del agente

Para definir la extensión del agente primero debemos agregar un atributo de paso `alpha`, que es un parámetro de entrada para la creación del bandido con valor de 0.1 por defecto.
Adicionalmente, eliminamos el atributo `occurrences` de la implementación del bandido, dado que en este caso no es necesario. De igual forma eliminamos el atributo `cummulative_rewards`, que será reemplazado por el atributo `episode_rewards`. Este último atributo debe guardar las recompensas obtenidas para cada episodio. Esto lo hacemos definiendo una lista de listas (inicialmente vacías), cada lista representando un brazo del agente.


In [70]:
#Implementación del algoritmo

import random
import math

class Bandit:
    def __init__(self, num_arms=10, epsilon=0.9, alpha=0.1):
        self.epsilon = epsilon
        self.alpha = alpha
        self.arms = [random.uniform(-3, 3) for _ in range(num_arms)]
        self.rewards = [0.0 for _ in range(num_arms)]
        self.episode_rewards = [[] for _ in range(num_arms)]

    def choose_arm(self) -> int:
        if random.random() < self.epsilon:
            return random.randrange(len(self.arms))

        max_reward = max(self.rewards)
        best_arms = [
            i for i, reward in enumerate(self.rewards)
            if reward == max_reward
        ]
        return random.choice(best_arms)

    def expected_reward(self, arm: int, episode: int) -> float:
        reward = self.arms[arm] * math.sin(episode)
        self.episode_rewards[arm].append(reward)
        return reward
    def calculate_expected_reward(self, arm: int) -> float:
        rewards = self.episode_rewards[arm]

        if not rewards:
            self.rewards[arm] = 0.0
            return 0.0

        alpha = self.alpha
        n = len(rewards)
        q1 = rewards[0]

        estimate = (1 - alpha) ** n * q1

        for i in range(1, n):
            estimate += (
                alpha
                * (1 - alpha) ** (n - i)
                * rewards[i]
            )

        self.rewards[arm] = estimate
        return estimate
    def run(self, episodes=1000) -> float:
        for episode in range(episodes):
            arm = self.choose_arm()
            self.expected_reward(arm, episode)
            self.calculate_expected_reward(arm)

        return self.rewards


In [71]:
#Pruebas de la definición del agente

bandit = Bandit()


In [72]:
#Pruebas adicionales
print("Número de brazos:", len(bandit.arms))
print("Epsilon:", bandit.epsilon)
print("Alpha:", bandit.alpha)
print("Recompensas iniciales:", bandit.rewards)
print("Número de historiales:", len(bandit.episode_rewards))

Número de brazos: 10
Epsilon: 0.9
Alpha: 0.1
Recompensas iniciales: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Número de historiales: 10


### Introducción de recompensas variables

El comportamiento de la función `expected_reward` requiere una división de sus responsabilidades. Por una parte, `expected_rewards` se encargará de calcular la recompensa obtenida para un brazo, dado por parámetro. La recompensa debe variar cada vez que se ejecuta un episodio. Para simular el cambio de la recompensa (dentro de un rango de valores conocidos) utilizaremos una función sinusoidal que nos da la recompensa obtenida del brazo como su recompensa inicial (i.e., la recompensa del brazo en la inicialización del bandido) multiplicado por el seno del paso actual. Es decir, para la iteración $i$, la recompensa del brazo $j$ es:

$$ r_i(j) = r_1(j)*\sin(i) $$

donde $r_1$ es la recompensa definida al crear el brazo, guardada en el atributo `arms`.

La recompensa se debe guardar (agregando el nuevo valor a la lista) en el atributo `episode_rewards` para el brazo dado. Este valor también debe ser retornado por la función.


In [73]:
# Pruebas de la función expected_rewards

bandit = Bandit()


In [74]:
reward = bandit.expected_reward(0, 1)

print("Recompensa obtenida:", reward)
print("Recompensa esperada:", bandit.arms[0] * math.sin(1))
print("Historial del brazo 0:", bandit.episode_rewards[0])


Recompensa obtenida: 2.1488019754040844
Recompensa esperada: 2.1488019754040844
Historial del brazo 0: [2.1488019754040844]


### Recompensa acumulada

Finalmente, definimos una nueva función `calculate_expected_reward`, que recibe un brazo dado por parámetro, y calcula la recompensa acumulada esperada del brazo según la fórmula de las recompensas no estacionarias $Q_i$. Este resultado se almacenará en el atributo `rewards` para el brazo.  
Note que en esta función la sumatoria debe realizarse desde 1 hasta n, sumando los primeros n-1 valores de la recompensa.

El estimado inicial ($Q_1$) va a ser la primera recompensa $r_1$.

La función debe retornar el valor calculado.

In [75]:
# Pruebas de la función calculated_expected_rewards

bandit = Bandit()


In [76]:
#Pruebas adicionales

# Probar la actualización de la recompensa estimada

r1 = bandit.expected_reward(0, 1)
q1 = bandit.calculate_expected_reward(0)

r2 = bandit.expected_reward(0, 2)
q2 = bandit.calculate_expected_reward(0)

print("Primera recompensa:", r1)
print("Primera estimación:", q1)
print("Segunda recompensa:", r2)
print("Estimación actualizada:", q2)


Primera recompensa: 0.8814120918947164
Primera estimación: 0.7932708827052447
Segunda recompensa: 0.9524579713415519
Estimación actualizada: 0.79966501185546


In [77]:
bandit = Bandit()


In [78]:
#Pruebas adicionales
# Prueba adicional de la recompensa acumulada

bandit = Bandit()

r1 = bandit.expected_reward(0, 1)
bandit.calculate_expected_reward(0)

r2 = bandit.expected_reward(0, 2)
estimacion = bandit.calculate_expected_reward(0)

print("Recompensas registradas:", bandit.episode_rewards[0])
print("Recompensa estimada final:", estimacion)


Recompensas registradas: [-2.1358038029976427, -2.307959439283137]
Recompensa estimada final: -1.937717429963573
